# 第 8 章 · 自主栈：从模块化流水线到端到端 VLA 与双系统

配套讲义：仓库里 `robot-system-architecture/notes/lessons/08-autonomy-stack.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

（各段耗时照动画；模块化串行那一条是练习题 1，代码里不算）

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第 5 步 / 动画 VLA | 等下一次重想 + 一次推理 | `reaction_time([0.02, 0.06, 0.04], 0.2, 0.30)` | 220 ms |
| 第 5 步 / 动画 S1 | 每 10 ms 看一次 | `reaction_time([0.008], 0.01, 0.30)` | 8 ms |
| 第 5 步 事件时刻不同 | 0.21 / 0.30 / 0.39 s | 同上 | 310 / 220 / 130 ms |
| 数学原理 平均等多久 | 均匀分布期望 $T/2$ | `mean_wait(0.2)` | 0.1001 s ≈ 0.1 s |
| 公式表 双系统 | $T_{S1} + t_{S1}$，和 $T_{S2}$ 无关 | `dual_system_react` | 都是 18 ms |

各段耗时照动画（VLA：编码 20 + VLM 60 + 动作头 40 ms，每 0.2 s 一圈；S1：8 ms，每 10 ms 一圈）。
模块化串行那条就是练习题 1，这里不算。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `reaction_time`**：动画的算法：第 c 圈从 c × 周期 开始，用开始那一刻的画面；事件之后开始的第一圈结束时才反应。

In [ ]:
def reaction_time(segments_s: list[float], period_s: float, event_s: float) -> float:
    """动画的算法：第 c 圈从 c × 周期 开始，用开始那一刻的画面；事件之后开始的第一圈结束时才反应。"""
    length = sum(segments_s)
    c = int(np.ceil(event_s / period_s - 1e-9))
    return c * period_s + length - event_s

**函数 `mean_wait`**：事件在一个周期里均匀出现，平均要等多久才轮到下一次采样（期望 T/2）。

In [ ]:
def mean_wait(period_s: float, n: int = 100_000, seed: int = 0) -> float:
    """事件在一个周期里均匀出现，平均要等多久才轮到下一次采样（期望 T/2）。"""
    rng = np.random.default_rng(seed)
    t = rng.uniform(0, period_s, n)
    return float((period_s - t).mean())

**函数 `dual_system_react`**：异步双系统：t_react ≈ T_S1 + t_S1，和 T_S2 无关（参数留着就是为了说明它没用上）。

In [ ]:
def dual_system_react(t_s1_period: float, t_s1_compute: float, t_s2_period: float) -> float:
    """异步双系统：t_react ≈ T_S1 + t_S1，和 T_S2 无关（参数留着就是为了说明它没用上）。"""
    return t_s1_period + t_s1_compute

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
event = 0.30
print("\n第五步 / 动画：第 0.30 秒杯子被碰走，多久之后开始执行新动作")
vla = reaction_time([0.02, 0.06, 0.04], 0.2, event)
s1 = reaction_time([0.008], 0.01, event)
print(f"  端到端 VLA（每 0.2 s 重想一次，一次 120 ms）：{vla * 1000:.0f} ms")
print(f"  双系统 S1（每 10 ms 一圈，一圈 8 ms）：{s1 * 1000:.0f} ms")

In [ ]:
print("\n第五步：事件发生的时刻不一样，VLA 的反应时间也不一样")
for ev in (0.21, 0.30, 0.39):
    print(f"  事件在 {ev:.2f} s → {reaction_time([0.02, 0.06, 0.04], 0.2, ev) * 1000:.0f} ms")

In [ ]:
print("\n数学原理：等下一次采样，平均等 T/2")
print(f"  T = 0.2 s：模拟平均等 {mean_wait(0.2):.4f} s，公式 T/2 = 0.1 s")

In [ ]:
print("\n公式表：异步双系统 t_react ≈ T_S1 + t_S1，和 T_S2 无关")
for t_s2 in (0.125, 0.5, 1.0):
    print(f"  T_S2 = {t_s2:.3f} s → 最坏 {dual_system_react(0.01, 0.008, t_s2) * 1000:.0f} ms")

## 3. 代码里才看得到的

- `- 1e-9` 是为了防浮点误差：0.30 / 0.01 在电脑里算出来是 30.000000000000004，不减一点 `ceil` 会多算一圈。
- 动画里 S1 的 8 ms 是"刚好在事件那一刻开始一圈"的最好情况；公式表的 18 ms 是"刚错过"的最坏情况。两个数都对，问的不是同一件事。

## 4. 自己改一改

- 把 VLA 的重想周期从 0.2 改成 0.1，看反应时间的范围怎么变。
- 用 `reaction_time` 自己编一条两段的流水线（比如 50 ms + 50 ms、每 100 ms 一圈），看最好和最坏情况各是多少。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 流水线可解释但串行慢，端到端省接口但要数据，双系统一个想一个做。
> 反应快慢看最快那条闭环；VLA 底下照样有控制和安全。